### 文搜图模型融合

In [1]:
import json

# 读取三个JSONL文件
file_paths = ["datasets/res1/test_predictions.jsonl","datasets/res1/test_predictions_h.jsonl"]
all_results = []

for file_path in file_paths:
    with open(file_path, 'r') as file:
        for line in file:
            result = json.loads(line)
            all_results.append(result)

# 合并候选结果
merged_results = {}
for result in all_results:
    text_id = result["text_id"]
    if text_id not in merged_results:
        merged_results[text_id] = {"image_ids": [], "counts": {}}
    merged_results[text_id]["image_ids"].extend(result["image_ids"])
    # 记录每个候选结果出现的次数
    for image_id in result["image_ids"]:
        merged_results[text_id]["counts"][image_id] = merged_results[text_id]["counts"].get(image_id, 0) + 1

# 投票融合
final_results = {}
for text_id, data in merged_results.items():
    image_counts = data["counts"]
    # 根据候选结果出现的次数进行排序
    sorted_counts = sorted(image_counts.items(), key=lambda x: x[1], reverse=True)
    # 选择前十个最可能的结果
    final_results[text_id] = [image_id for image_id, count in sorted_counts[:10]]

# 输出结果为JSONL文件
output_file = "datasets/res1/merged_test_predictions.jsonl"
with open(output_file, 'w') as file:
    for text_id, image_ids in final_results.items():
        result = {"text_id": text_id, "image_ids": image_ids}
        file.write(json.dumps(result) + '\n')


### 图搜文模型融合

In [4]:
import json

# 读取三个JSONL文件
file_paths = ["datasets/res2/test_predictions_tr.jsonl","datasets/res2/test_predictions_tr_h.jsonl"]
all_results = []

for file_path in file_paths:
    with open(file_path, 'r') as file:
        for line in file:
            result = json.loads(line)
            all_results.append(result)

# 合并候选结果
merged_results = {}
for result in all_results:
    image_id = result["image_id"]
    if image_id not in merged_results:
        merged_results[image_id] = {"text_ids": [], "counts": {}}
    merged_results[image_id]["text_ids"].extend(result["text_ids"])
    # 记录每个候选结果出现的次数
    for text_id in result["text_ids"]:
        merged_results[image_id]["counts"][text_id] = merged_results[image_id]["counts"].get(text_id, 0) + 1

# 投票融合
final_results = {}
for image_id, data in merged_results.items():
    image_counts = data["counts"]
    # 根据候选结果出现的次数进行排序
    sorted_counts = sorted(image_counts.items(), key=lambda x: x[1], reverse=True)
    # 选择前十个最可能的结果
    final_results[image_id] = [text_id for text_id, count in sorted_counts[:10]]

# 输出结果为JSONL文件
output_file = "datasets/res2/merged_test_predictions.jsonl"
with open(output_file, 'w') as file:
    for image_id, text_ids in final_results.items():
        result = {"image_id": image_id, "text_ids": text_ids}
        file.write(json.dumps(result) + '\n')

### 结果按重命名处理，返回result1.csv

In [5]:
import pandas as p
# 读取JSONL文件
def read_jsonl(file_path):
    data = []
    with open(file_path, 'r') as file:
        for line in file:
            data.append(eval(line))
    return data
# 读取CSV文件
def read_csv(file_path):
    return pd.read_csv(file_path)
# 根据text_id_mapping.csv替换text_id
def replace_text_id(text_id, mapping_df):
    original_text_id = mapping_df[mapping_df['text_id'] == text_id]['original_text_id']
    return original_text_id.values[0] if not original_text_id.empty else None
# 根据renamed_files.csv替换image_id
def replace_image_id(image_ids, renamed_df):
    renamed_image_ids = []
    for image_id in image_ids:
        renamed_image_id = renamed_df[renamed_df['Renamed Name'].str.startswith(str(image_id) + '.')]['Original Name']
        if not renamed_image_id.empty:
            renamed_image_ids.append(renamed_image_id.values[0])
    return renamed_image_ids
# 处理jsonl文件和CSV文件
def process_data(jsonl_file, renamed_file, mapping_file):
    # 读取数据
    jsonl_data = read_jsonl(jsonl_file)
    renamed_df = read_csv(renamed_file)
    mapping_df = read_csv(mapping_file)
    result = []
    # 处理每一行数据
    for row in jsonl_data:
        text_id = row['text_id']
        image_ids = row['image_ids'][:5]  # 取前五个
        original_text_id = replace_text_id(text_id, mapping_df)
        if original_text_id is not None:
            original_image_ids = replace_image_id(image_ids, renamed_df)
            for i, image_id in enumerate(original_image_ids):
                result.append([original_text_id, i+1, image_id])
    # 创建DataFrame
    df = pd.DataFrame(result, columns=['text_id', 'similarity_ranking', 'result_image_id'])
    # 保存CSV文件
    df.to_csv('data_csv/res1/result1.csv', index=False)
# 调用函数处理数据
process_data('datasets/res1/merged_test_predictions.jsonl', 'data_csv/res1/renamed_files.csv', 'data_csv/res1/text_id_mapping.csv')

### 结果按重命名处理，返回result2.csv

In [8]:
import pandas as pd
# 读取JSONL文件
def read_jsonl(file_path):
    data = []
    with open(file_path, 'r') as file:
        for line in file:
            data.append(eval(line))
    return data
# 读取CSV文件
def read_csv(file_path):
    return pd.read_csv(file_path)
# 根据text_id_mapping.csv替换text_id
def replace_text_id(text_ids, mapping_df):
    replaced_text_ids = []
    for text_id in text_ids[:5]:  # 取前五个
        replaced_text_id = mapping_df[mapping_df['text_id'] == text_id]['original_text_id']
        if not replaced_text_id.empty:
            replaced_text_ids.append(replaced_text_id.values[0])
    return replaced_text_ids
# 根据renamed_files.csv替换image_id
def replace_image_id(image_id, renamed_df):
    original_image_id = renamed_df[renamed_df['Renamed Name'].str.startswith(str(image_id) + '.')]['Original Name']
    return original_image_id.values[0] if not original_image_id.empty else None
# 处理jsonl文件和CSV文件
def process_data(jsonl_file, renamed_file, mapping_file):
    # 读取数据
    jsonl_data = read_jsonl(jsonl_file)
    renamed_df = read_csv(renamed_file)
    mapping_df = read_csv(mapping_file)
    result = []
    # 处理每一行数据
    for row in jsonl_data:
        image_id = row['image_id']
        text_ids = row['text_ids']
        original_image_id = replace_image_id(image_id, renamed_df)
        if original_image_id is not None:
            original_text_ids = replace_text_id(text_ids, mapping_df)
            for i, text_id in enumerate(original_text_ids):
                result.append([original_image_id, i+1, text_id])
    # 创建DataFrame
    df = pd.DataFrame(result, columns=['image_id', 'similarity_ranking', 'result_text_id'])
    df.to_csv('data_csv/res2/result2.csv', index=False)
# 调用函数处理数据
process_data('datasets/res2/merged_test_predictions.jsonl', 'data_csv/res2/renamed_files.csv', 'data_csv/res2/text_id_mapping.csv')